# 🚜 Kaggriculture: Autonomous AI Farming Grandmaster Agent
### High-Efficiency Multi-Agent Coordination Engine for the Kaggle Kaggriculture Competition

---

## 1. Executive Summary & Strategic Architecture

In **Kaggriculture**, two autonomous AI agents manage separate competitive farms over a 30-day season (**720 turns**, 24 turns/day). The sole victory condition is **total coins in the bank at turn 719**. Unsold crops, livestock, or shed inventory at step 720 are worth **$0**.

### 1.1 The Six Engineering Pillars
1. **The Stored Programme Paradigm (Action Tapes)**:
   - Rather than making myopic, greedy turn-by-turn decisions that lead to spatial deadlocks and worker collisions, our agent employs **precomputed, multi-agent coordinating action tapes**.
   - These tapes encode synchronized worker movement, planting patterns, daily watering, livestock feeding/care, and shed drops across 30 days.

2. **Public Checkpoint Routing at Turn 144 (Day 6)**:
   - Town shops unlock every 3 days (Day 3, 6, 9, 12, 15, 18, 21, 24).
   - At Turn 144 (Day 6), the first two shop instances are known. The agent identifies the shop pair, forecasts demand drains across the market, and locks into the highest-ROI action tape for the remainder of the season.

3. **Dynamic Elasticity & Front-Running**:
   - Market prices move dynamically per commodity:
     $$\text{price}(inv) = \text{base} + \text{sign} \cdot \text{amp} \cdot f(|inv - I_0|)$$
   - Premium crops (**Strawberry**, **Melon**) experience steep price drops under gluts ($f(x) = x^2$ or linear with target $>1.0$).
   - The agent quotes prices ahead of opponent selling, prioritizing scarce items and scheduling dumps before market crashes.

4. **Shed Capacity Hard Cap (100 Items)**:
   - Non-seed inventory across the entire farm is capped at 100 in the shed. Any overflow at end-of-day or mid-day is **silently discarded**.
   - Flow control mechanisms dynamically throttle harvesting and flush inventory to market before overflow can occur.

5. **Livestock Feed Invariant**:
   - Animals (Geese, Cows, Sheep) produce recurring high-margin products (Eggs, Milk, Wool) and fertilizer, but require **1 Wheat per day**.
   - Missing 2 consecutive days causes permanent escape. The agent maintains an active emergency wheat reserve and auto-replenishes from the market if grown wheat is insufficient.

6. **Terminal Shadow Tree Search (Turns 712–719)**:
   - On Days 29–30, cyclic farming halts. A 64-rollout shadow tree search simulates farmer movements to harvest all ripe crops and bring them to the shed.
   - At turn 718, a mandatory 100% liquidation order converts all remaining assets to cash.


## 2. Environment Setup & Dependency Verification

In [ ]:
import sys
import os
import json
from pathlib import Path

# Verify python version and environment
print(f"Python Version: {sys.version.split()[0]}")
try:
    import kaggle_environments
    print(f"kaggle-environments Version: {kaggle_environments.__version__}")
except ImportError:
    print("Installing kaggle-environments...")
    !pip install -q kaggle-environments
    import kaggle_environments
    print(f"kaggle-environments Version: {kaggle_environments.__version__}")


## 3. Production Grandmaster Agent (`main.py`)

The cell below writes the complete, standalone agent code into `main.py`. It requires no external dependencies beyond the standard library, guaranteeing sub-second execution on Kaggle evaluation sandboxes.


In [ ]:
import base64, zlib
from pathlib import Path

# Byte-exact payload of Grandmaster V55 + Smaller Market Shock + Layer D Order Book Engine (1,025,628 bytes)
# sha256: a701d59e5d53eac6fb57e5766a8ae88a7a8ceac8e97697ff870f99d019fa4711
_PAYLOAD = (
    "..."
)

Path("main.py").write_bytes(zlib.decompress(base64.b64decode(_PAYLOAD)))
print("Wrote main.py:", f"{Path('main.py').stat().st_size:,} bytes")


## 4. AST Integrity & Syntax Verification

In [ ]:
import ast, hashlib
from pathlib import Path

source = Path('main.py').read_bytes()
EXPECTED_SHA256 = 'a701d59e5d53eac6fb57e5766a8ae88a7a8ceac8e97697ff870f99d019fa4711'
actual_sha = hashlib.sha256(source).hexdigest()
assert actual_sha == EXPECTED_SHA256, f'Hash mismatch: {actual_sha}'
tree = ast.parse(source.decode('utf-8'))

# Verify Kaggle entry point (last callable in namespace)
namespace = {"__name__": "main"}
exec(compile(source, "main.py", "exec"), namespace)
callables = [v for v in namespace.values() if callable(v)]
entry = callables[-1]
assert entry.__name__ == "_cxd_agent", f"Unexpected entry point: {entry.__name__}"
print(f"[OK] AST verified successfully: {len(tree.body)} top-level statements, {len(source):,} bytes")
print(f"[OK] Entry point Kaggle will execute: {entry.__name__} (Order Book Lockstep Settlement Engine)")


## 5. Build Submission Archive (`submission.tar.gz`)

In [ ]:
import gzip, hashlib, io, tarfile
from pathlib import Path

source = Path("main.py").read_bytes()
EXPECTED_SHA256 = 'a701d59e5d53eac6fb57e5766a8ae88a7a8ceac8e97697ff870f99d019fa4711'
assert hashlib.sha256(source).hexdigest() == EXPECTED_SHA256
buffer = io.BytesIO()
with tarfile.open(fileobj=buffer, mode="w:gz", format=tarfile.GNU_FORMAT) as tar:
    info = tarfile.TarInfo("main.py")
    info.size, info.mtime, info.mode = len(source), 0, 0o644
    tar.addfile(info, io.BytesIO(source))

tar_bytes = buffer.getvalue()
Path("submission.tar.gz").write_bytes(tar_bytes)
print(f"[OK] Created deterministic submission.tar.gz: {len(tar_bytes):,} bytes")
print(f"[OK] Archive SHA256: {hashlib.sha256(tar_bytes).hexdigest()}")
